# Machine Learning Pipeline: Understanding `fit()`, `transform()`, `fit_transform()`, and Related Methods

## In scikit-learn, almost every object belongs to one of two categories:

| Category | Purpose | Examples | Common Methods |
|----------|---------|----------|----------------|
| **Transformers** | Change or preprocess data into a different representation. | `StandardScaler`, `MinMaxScaler`, `OneHotEncoder`, `PCA`, `PolynomialFeatures` | `fit()`, `transform()`, `fit_transform()`, `inverse_transform()`|
| **Estimators (Models)** | Learn patterns from data and make predictions. | `LinearRegression`, `LogisticRegression`, `RandomForest`, `SVM`, `DecisionTree` | `fit()`, `predict()`, `predict_proba()`, `score()` |

In [40]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [41]:
# Generate consistent fake data
# 1. Create a dummy dataset (100 samples, 2 features)
# Imagine X contains house size and age, and y is 1 if it sold fast, 0 if not.
np.random.seed(42)
sizes = np.random.randint(500, 3500, size=100)  # Large numbers
ages = np.random.randint(1, 50, size=100)  # Small numbers
X = np.column_stack((sizes, ages))
y = np.random.choice([0, 1], size=100)
# Split into Train (80%) and Test (20%) sets
# train_test_split() is flexible. It accepts Pandas DataFrames/Series, NumPy arrays, and even Python lists. The output type matches the input type.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("--- Raw Training Data (First 5 rows) ---")
print(pd.DataFrame(X_train, columns=["Size (sqft)", "Age (years)"]).head())

--- Raw Training Data (First 5 rows) ---
   Size (sqft)  Age (years)
0         3445           23
1         1291            6
2         1247           40
3         1146            9
4         2000           45


## Understanding `fit()`

### 1. The `fit()` Function (Learning Phase)
*   **What it does:** The tool analyzes the dataset to calculate and store internal parameters. For data scaling, it calculates the **mean** and **standard deviation**. 
*   **Crucial Concept:** It **does not alter your data**. It simply takes notes.
*   *Analogy:* A student reads a textbook and memorizes the formulas, but hasn't written anything down on an exam paper yet.

In [48]:
# Initialize the scaler
scaler = StandardScaler()

# Run fit() on our training data
scaler.fit(X_train)

print("--- What did fit() actually do? ---")
print(f"It looked at the data and calculated the averages!")
print(f"Learned Mean for Size: {scaler.mean_[0]:.2f}")
print(f"Learned Mean for Age:  {scaler.mean_[1]:.2f}\n")

print("--- Did the original data change? ---")
print("No! Firsr 5 row (Still the raw value)")
print(pd.DataFrame(X_train, columns=["Size (sqft)", "Age (years)"]).head())

--- What did fit() actually do? ---
It looked at the data and calculated the averages!
Learned Mean for Size: 1998.55
Learned Mean for Age:  25.45

--- Did the original data change? ---
No! Firsr 5 row (Still the raw value)
   Size (sqft)  Age (years)
0         3445           23
1         1291            6
2         1247           40
3         1146            9
4         2000           45


## Understanding `transform()`

### 2. The `transform()` Function (Application Phase)
*   **What it does:** The tool uses the notes it took during `fit()` to actually modify and scale the dataset. 
*   **Crucial Concept:** You *must* run `fit()` before you can run `transform()`. We use it on both the training set and the testing set to make sure all data is converted using the exact same mathematical rules.
*   *Analogy:* The student uses the formulas they memorized during the `fit()` phase to actually solve the math problems on the page.

In [43]:
# Transform the training data using the averages learned in the previous cell
X_train_scaled = scaler.transform(X_train)

# Transform the testing data using those EXACT SAME training averages (Prevents cheating/leakage!)
X_test_scaled = scaler.transform(X_test)

print("--- Transformed Training Data (First 5 rows) ---")
print(pd.DataFrame(X_train_scaled, columns=["Size (Scaled)", "Age (Scaled)"]).head())
print(
    "\nNotice that the numbers have now been compressed safely between roughly -3 and +3!"
)

--- Transformed Training Data (First 5 rows) ---
   Size (Scaled)  Age (Scaled)
0       1.679438     -0.176665
1      -0.821519     -1.402505
2      -0.872606      1.049175
3      -0.989875     -1.186181
4       0.001684      1.409716

Notice that the numbers have now been compressed safely between roughly -3 and +3!


In [44]:
# Generate consistent fake data
# 1. Create a dummy dataset (100 samples, 2 features)
# Imagine X contains house size and age, and y is 1 if it sold fast, 0 if not.
np.random.seed(42)
sizes = np.random.randint(500, 3500, size=100)  # Large numbers
ages = np.random.randint(1, 50, size=100)  # Small numbers
X = np.column_stack((sizes, ages))
y = np.random.choice([0, 1], size=100)
# Split into Train (80%) and Test (20%) sets
# train_test_split() is flexible. It accepts Pandas DataFrames/Series, NumPy arrays, and even Python lists. The output type matches the input type.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("--- Raw Training Data (First 5 rows) ---")
print(pd.DataFrame(X_train, columns=["Size (sqft)", "Age (years)"]).head())

--- Raw Training Data (First 5 rows) ---
   Size (sqft)  Age (years)
0         3445           23
1         1291            6
2         1247           40
3         1146            9
4         2000           45


## Understanding `fit_transform()`

### 3. The `fit_transform()` Function (The Shortcut)
*   **What it does:** It runs `fit()` and `transform()` sequentially in a single line of code. It learns the parameters *and* immediately returns the scaled data.
*   **When to use it:** You use this **only on your training data** to speed up your workflow. You never use it on test data (because you don't want your scaler learning anything new from your test set).

In [45]:
# This single line replaces running scaler.fit() and scaler.transform() separately
shortcut_scaler = StandardScaler()
X_train_shortcut = shortcut_scaler.fit_transform(X_train)

print("--- Verification Check after fit_transform ---")
print("--- Transformed Training Data (First 5 rows) ---")
print(pd.DataFrame(X_train_shortcut, columns=["Size (Scaled)", "Age (Scaled)"]).head())
print(
    "\nNotice that the numbers have now been compressed safely between roughly -3 and +3!"
)

--- Verification Check after fit_transform ---
--- Transformed Training Data (First 5 rows) ---
   Size (Scaled)  Age (Scaled)
0       1.679438     -0.176665
1      -0.821519     -1.402505
2      -0.872606      1.049175
3      -0.989875     -1.186181
4       0.001684      1.409716

Notice that the numbers have now been compressed safely between roughly -3 and +3!


## Understanding `predict()`

### 4. The `predict()` Function (The Decision Phase)
*   **What it does:** This belongs to **Machine Learning models** (like Logistic Regression), not data scalers. After the model uses `fit()` to learn the mathematical relationships between your features and your labels, `predict()` takes new, unseen features and outputs its final guess.
*   *Analogy:* The student takes the final exam on a brand-new page and writes down their final answers.

In [46]:
# Initialize the model
model = LogisticRegression()

# Step 1: Model uses fit() to learn the relationship between scaled features and y_train
model.fit(X_train_scaled, y_train)

# Step 2: Model uses predict() to make guesses on the scaled test data
predictions = model.predict(X_test_scaled)

# Step 3: Model uses predict_proba() to show its percentage confidence
probabilities = model.predict_proba(X_test_scaled)

# Combine into a final output table for clear reading
output_analysis = pd.DataFrame(
    {
        "Actual House Outcome": y_test,
        "Model Prediction": predictions,
        "Confidence (Will Sell Fast)": probabilities[:, 1],
    }
)

print("--- Final Model Evaluation (First 5 Test Houses) ---")
print(output_analysis.head())

--- Final Model Evaluation (First 5 Test Houses) ---
   Actual House Outcome  Model Prediction  Confidence (Will Sell Fast)
0                     1                 0                     0.407533
1                     0                 0                     0.364614
2                     1                 1                     0.520953
3                     0                 0                     0.418611
4                     0                 0                     0.308168


## Understanding `score()`

### 5. The `score()` Function (The Grading Phase)
*   **What it does:** This belongs to **Machine Learning models** (like Logistic Regression or Linear Regression). Instead of giving you raw predictions, `score()` automatically evaluates how well the model performed by comparing its predictions against the actual, true answers. 
*   **What numbers does it return?**
    *   For **Classification models** (like our house sale predictor): It returns **Accuracy** (the percentage of correct guesses between `0.0` and `1.0`).
    *   For **Regression models** (like predicting a exact house price): It returns the **$R^2$ score** (R-squared, measuring how well the model explains the variance in the data).
*   *Analogy:* The teacher grades the final exam paper and returns a percentage score (e.g., 85% correct).

In [47]:
# Evaluate performance on the Training Set (Should be high because the model practiced on this)
train_accuracy = model.score(X_train_scaled, y_train)

# Evaluate performance on the Testing Set (The true test of how well the model generalizes to new data)
test_accuracy = model.score(X_test_scaled, y_test)

print("--- Model Performance Report ---")
print(f"Training Accuracy: {train_accuracy * 100:.1f}%")
print(f"Testing Accuracy:  {test_accuracy * 100:.1f}%")

print("\nUnder the hood, model.score(X_test_scaled, y_test) runs model.predict() ")
print("automatically and compares those guesses directly to the true y_test values!")

--- Model Performance Report ---
Training Accuracy: 66.2%
Testing Accuracy:  65.0%

Under the hood, model.score(X_test_scaled, y_test) runs model.predict() 
automatically and compares those guesses directly to the true y_test values!
